# 🩺 QLoRA Fine-tuning: Mistral-7B for Medical Question Answering

**Method**: QLoRA (4-bit quantisation + LoRA adapters via PEFT)  
**Base model**: `mistralai/Mistral-7B-Instruct-v0.2`  
**Dataset**: ChatDoctor-HealthCareMagic-100k (medical Q&A)  
**Tracking**: Weights & Biases  
**Deployment**: HuggingFace Hub + Spaces

> ⚙️ **Runtime**: Kaggle / Google Colab with **T4 GPU** (free tier)  
> 💾 GPU memory required: ~14 GB (fits in T4's 16 GB with 4-bit quant)

---

## 0 · Environment Setup

In [ ]:
# Install all dependencies
!pip install -q \
    transformers==4.38.2 \
    datasets==2.17.1 \
    peft==0.9.0 \
    trl==0.8.1 \
    bitsandbytes==0.42.0 \
    accelerate==0.27.2 \
    evaluate==0.4.1 \
    rouge_score \
    sacrebleu \
    wandb \
    gradio

print('✅ All packages installed')

In [ ]:
import os, json, torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# GPU check
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Authenticate — set your tokens as Kaggle secrets or Colab secrets
import wandb
from huggingface_hub import login

# Option A: Kaggle secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN    = secrets.get_secret('HF_TOKEN')
WANDB_TOKEN = secrets.get_secret('WANDB_TOKEN')
HF_USERNAME = secrets.get_secret('HF_USERNAME')

# Option B: Colab secrets (uncomment)
# from google.colab import userdata
# HF_TOKEN    = userdata.get('HF_TOKEN')
# WANDB_TOKEN = userdata.get('WANDB_TOKEN')
# HF_USERNAME = userdata.get('HF_USERNAME')

login(token=HF_TOKEN)
wandb.login(key=WANDB_TOKEN)
print('✅ Authenticated')

---
## 1 · Data Preparation

We load the **ChatDoctor-HealthCareMagic-100k** dataset — 100 k real patient-doctor conversations — and format each example into the Mistral instruction template:

```
<s>[INST] {system_prompt}\n{question} [/INST] {answer} </s>
```

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME   = 'mistralai/Mistral-7B-Instruct-v0.2'
MAX_LENGTH   = 512
SYSTEM_PROMPT = (
    'You are a knowledgeable medical assistant. '
    'Answer the following medical question accurately and clearly.\n\n'
)

# Load & subsample (remove [:20000] for full run)
raw = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', split='train')
raw = raw.shuffle(seed=42).select(range(20_000))
print(f'Loaded {len(raw):,} examples')
print('Columns:', raw.column_names)
print('\nSample row:')
raw[0]

In [ ]:
def format_example(row):
    question = row.get('input') or row.get('question', '')
    answer   = row.get('output') or row.get('answer',   '')
    text = (
        f'<s>[INST] {SYSTEM_PROMPT}'
        f'{question.strip()} [/INST] '
        f'{answer.strip()} </s>'
    )
    return {'text': text}

formatted = raw.map(format_example, remove_columns=raw.column_names)
formatted = formatted.filter(lambda x: len(x['text']) > 100)

# Train / val split (90 / 10)
split   = formatted.train_test_split(test_size=0.10, seed=42)
dataset = DatasetDict({'train': split['train'], 'validation': split['test']})

print(f'Train : {len(dataset["train"]):,}')
print(f'Val   : {len(dataset["validation"]):,}')

# Show 2 examples
for i in range(2):
    print(f'\n─── Example {i+1} ───')
    print(dataset['train'][i]['text'][:500], '…')

In [ ]:
# Token-length distribution
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

sample = dataset['train'].select(range(2000))
lengths = [len(tokenizer(ex['text'])['input_ids']) for ex in sample]

print(f'Min    : {min(lengths)}')
print(f'Max    : {max(lengths)}')
print(f'Mean   : {np.mean(lengths):.0f}')
print(f'Median : {np.median(lengths):.0f}')
print(f'P90    : {np.percentile(lengths, 90):.0f}')
print(f'> {MAX_LENGTH}: {sum(l > MAX_LENGTH for l in lengths)} ({100*sum(l > MAX_LENGTH for l in lengths)/len(lengths):.1f}%)')

plt.figure(figsize=(10, 4))
plt.hist(lengths, bins=50, color='#4F86C6', edgecolor='white', alpha=0.85)
plt.axvline(MAX_LENGTH, color='#E63946', linewidth=2, linestyle='--', label=f'max_length={MAX_LENGTH}')
plt.xlabel('Token Length')
plt.ylabel('Count')
plt.title('Training Example Token-Length Distribution (2k sample)')
plt.legend()
plt.tight_layout()
plt.savefig('length_distribution.png', dpi=150)
plt.show()

---
## 2 · Model Setup — 4-bit QLoRA

### Why QLoRA?

Full fine-tuning a 7B model requires ~56 GB of GPU memory (fp32) — impossible on free tiers. QLoRA solves this with two tricks:

1. **4-bit NF4 quantisation** (BitsAndBytes) — compresses base model weights from 16-bit to 4-bit, reducing VRAM from ~14 GB → ~4 GB
2. **LoRA adapters** (PEFT) — instead of updating all 7B params, we inject small trainable rank-decomposition matrices into the attention layers only (~85M params, ~1.2% of total)

During training, gradients only flow through the LoRA adapters (in fp16), while the frozen base model stays in 4-bit. After training, adapters are merged back into the base weights at full precision.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 4-bit BitsAndBytes config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',        # NormalFloat4
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,          # nested quant → saves ~0.4 GB
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = 'auto',
    torch_dtype         = torch.float16,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

print(f'GPU memory after loading base: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Prepare for k-bit training (casts LayerNorm to fp32, enables grad checkpointing)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA config
lora_config = LoraConfig(
    r              = 64,          # rank — higher = more capacity
    lora_alpha     = 16,          # scaling factor (effective lr = alpha/r)
    lora_dropout   = 0.1,
    target_modules = ['q_proj', 'v_proj'],  # key attention projections
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params : {trainable:,}  ({100*trainable/total:.4f}%)')
print(f'Total params     : {total:,}')
print(f'GPU memory now   : {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## 3 · Training

In [ ]:
import wandb
from transformers import TrainingArguments, TrainerCallback
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

wandb.init(
    project = 'mistral-medical-qlora',
    tags    = ['mistral-7b', 'qlora', 'medical-qa'],
    config  = {
        'lora_r': 64, 'lora_alpha': 16, 'lora_dropout': 0.1,
        'epochs': 3, 'lr': 2e-4, 'batch_size': 4,
        'grad_accum': 4, 'max_seq_length': MAX_LENGTH,
    }
)
print(f'W&B run: {wandb.run.url}')

In [ ]:
class GPUMemoryCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available() and logs is not None:
            logs['gpu_memory_gb'] = torch.cuda.memory_allocated() / 1e9

training_args = TrainingArguments(
    output_dir                  = 'checkpoints/mistral-medical-qlora',
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = 4,       # effective batch = 16
    learning_rate               = 2e-4,
    warmup_ratio                = 0.03,
    lr_scheduler_type           = 'cosine',
    fp16                        = True,
    logging_steps               = 10,
    eval_steps                  = 100,
    save_steps                  = 200,
    evaluation_strategy         = 'steps',
    save_strategy               = 'steps',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    report_to                   = 'wandb',
    gradient_checkpointing      = True,
    optim                       = 'paged_adamw_32bit',
    max_grad_norm               = 0.3,
    group_by_length             = True,
)

# Only compute loss on answer tokens (not the [INST] prompt)
collator = DataCollatorForCompletionOnlyLM(
    response_template = ' [/INST]',
    tokenizer         = tokenizer,
)

trainer = SFTTrainer(
    model              = model,
    args               = training_args,
    train_dataset      = dataset['train'],
    eval_dataset       = dataset['validation'],
    tokenizer          = tokenizer,
    data_collator      = collator,
    dataset_text_field = 'text',
    max_seq_length     = MAX_LENGTH,
    callbacks          = [GPUMemoryCallback()],
)

print('🚀 Starting training …')
trainer_output = trainer.train()
trainer.save_model('checkpoints/mistral-medical-qlora')
tokenizer.save_pretrained('checkpoints/mistral-medical-qlora')
wandb.finish()
print('✅ Training complete')

In [ ]:
# Plot training loss from trainer log history
log_history = trainer.state.log_history

train_steps  = [l['step']       for l in log_history if 'loss' in l]
train_losses = [l['loss']       for l in log_history if 'loss' in l]
eval_steps   = [l['step']       for l in log_history if 'eval_loss' in l]
eval_losses  = [l['eval_loss']  for l in log_history if 'eval_loss' in l]

plt.figure(figsize=(12, 4))
plt.plot(train_steps, train_losses, label='Train Loss', color='#4F86C6', linewidth=1.5)
plt.plot(eval_steps,  eval_losses,  label='Val Loss',   color='#E63946', linewidth=2, marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('QLoRA Training — Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('Training metrics:', trainer_output.metrics)

---
## 4 · Evaluation — ROUGE + BLEU

In [ ]:
from evaluate import load as load_metric
from peft import PeftModel

rouge_metric = load_metric('rouge')
bleu_metric  = load_metric('bleu')

NUM_TEST = 50
test_set = dataset['validation'].shuffle(seed=99).select(range(NUM_TEST))

def parse_qa(text):
    try:
        q = text.split('[INST]')[1].split('[/INST]')[0]
        q = q.replace(SYSTEM_PROMPT, '').strip()
        a = text.split('[/INST]')[1].replace('</s>', '').strip()
        return q, a
    except:
        return '', ''

pairs = [parse_qa(ex['text']) for ex in test_set]
pairs = [(q, a) for q, a in pairs if q and a]
print(f'Evaluating on {len(pairs)} samples')

In [ ]:
@torch.no_grad()
def generate(mdl, question, max_new_tokens=256, temperature=0.3):
    prompt  = f'<s>[INST] {SYSTEM_PROMPT}{question.strip()} [/INST]'
    inputs  = tokenizer(prompt, return_tensors='pt').to(mdl.device)
    outputs = mdl.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature, do_sample=True,
        top_p=0.9, repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# Fine-tuned model is already in `model`
print('Generating fine-tuned responses …')
ft_preds = [generate(model, q) for q, _ in tqdm(pairs)]

# Load base model for comparison
print('Loading base model for comparison …')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.float16,
)
base_model.eval()
print('Generating base model responses …')
base_preds = [generate(base_model, q) for q, _ in tqdm(pairs)]
del base_model; torch.cuda.empty_cache()

In [ ]:
ground_truths = [a for _, a in pairs]
questions     = [q for q, _ in pairs]

def compute_metrics(preds, refs):
    r = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    b = bleu_metric.compute(predictions=[p.split() for p in preds],
                            references=[[r.split()] for r in refs])
    return {
        'rouge1': round(r['rouge1']*100, 2),
        'rouge2': round(r['rouge2']*100, 2),
        'rougeL': round(r['rougeL']*100, 2),
        'bleu':   round(b['bleu']*100,   2),
    }

base_metrics = compute_metrics(base_preds, ground_truths)
ft_metrics   = compute_metrics(ft_preds,   ground_truths)

print(f'\n{"Metric":<10} {"Base Model":>12} {"Fine-tuned":>12} {"Δ":>8}')
print('─' * 46)
for k in ['rouge1', 'rouge2', 'rougeL', 'bleu']:
    d = ft_metrics[k] - base_metrics[k]
    print(f'{k:<10} {base_metrics[k]:>11.2f}% {ft_metrics[k]:>11.2f}% {("+" if d>=0 else "")}{d:.2f}%')

In [ ]:
# Side-by-side comparison table (first 10)
import pandas as pd

table = pd.DataFrame({
    'Question':            [q[:80]+'…' for q in questions[:10]],
    'Base Model Response': [b[:120]+'…' for b in base_preds[:10]],
    'Fine-tuned Response': [f[:120]+'…' for f in ft_preds[:10]],
    'Ground Truth':        [g[:120]+'…' for g in ground_truths[:10]],
})
pd.set_option('display.max_colwidth', 130)
table

In [ ]:
# Save evaluation report
import os; os.makedirs('results', exist_ok=True)

report = {
    'num_samples': len(pairs),
    'base_model_metrics':     base_metrics,
    'finetuned_model_metrics': ft_metrics,
    'full_comparison_table': [
        {'question': q, 'base_response': b, 'finetuned_response': f, 'ground_truth': g}
        for q, b, f, g in zip(questions, base_preds, ft_preds, ground_truths)
    ]
}
with open('results/evaluation_report.json', 'w') as fh:
    json.dump(report, fh, indent=2)
print('✅ Report saved → results/evaluation_report.json')

---
## 5 · Merge & Push to HuggingFace Hub

In [ ]:
HUB_REPO_ID = f'{HF_USERNAME}/mistral-7b-medical-qa-qlora'

print('Merging LoRA weights into base model …')
# Reload base in fp16 for clean merge
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
merged = PeftModel.from_pretrained(base_for_merge, 'checkpoints/mistral-medical-qlora')
merged = merged.merge_and_unload()
merged.eval()

print(f'Pushing to hub: {HUB_REPO_ID}')
merged.push_to_hub(HUB_REPO_ID, use_auth_token=True, safe_serialization=True)
tokenizer.push_to_hub(HUB_REPO_ID, use_auth_token=True)
print(f'✅ Model live at: https://huggingface.co/{HUB_REPO_ID}')

---
## 6 · Quick Inference Test

In [ ]:
test_questions = [
    'What are the symptoms of type 2 diabetes?',
    'How does high blood pressure damage the kidneys?',
    'What is the difference between viral and bacterial pneumonia?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {generate(merged, q)}')
    print()